In [1]:

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from feature_engine.outliers import Winsorizer
import joblib


In [2]:
# Load dataset
dataset = pd.read_csv(r"CarswithEnginetype.csv")


In [3]:
dataset.head()

,MPG,Enginetype,HP,VOL,SP,WT
0,53.700681,petrol,49.0,89.0,104.185353,28.762059
1,50.013401,hybrid,55.0,92.0,105.461264,30.466833
2,50.013401,diesel,55.0,92.0,105.461264,30.193597
3,45.696322,lpg,70.0,92.0,113.461264,30.632114
4,50.504232,petrol,53.0,92.0,104.461264,29.889149


In [5]:
dataset.corr(numeric_only=True)

,MPG,HP,VOL,SP,WT
MPG,1.000000,-0.724239,-0.532726,-0.677916,-0.524217
HP,-0.724239,1.000000,0.063045,0.972857,0.069146
VOL,-0.532726,0.063045,1.000000,0.083075,0.999205
SP,-0.677916,0.972857,0.083075,1.000000,0.089692
WT,-0.524217,0.069146,0.999205,0.089692,1.000000


In [22]:
# Separate features and target
X = dataset.iloc[:, 1:]  # Features
y = dataset.iloc[:, 0]   # Target


In [23]:
X.head()

,Enginetype,HP,VOL,SP,WT
0,petrol,49.0,89.0,104.185353,28.762059
1,hybrid,55.0,92.0,105.461264,30.466833
2,diesel,55.0,92.0,105.461264,30.193597
3,lpg,70.0,92.0,113.461264,30.632114
4,petrol,53.0,92.0,104.461264,29.889149


In [24]:
y.head()

0    53.700681
1    50.013401
2    50.013401
3    45.696322
4    50.504232
Name: MPG, dtype: float64

In [25]:
# Identify categorical and numerical columns
numeric_features = X.select_dtypes(exclude=['object']).columns
categorical_features = X.select_dtypes(include=['object']).columns


In [26]:
numeric_features

Index(['HP', 'VOL', 'SP', 'WT'], dtype='object')

In [27]:
categorical_features

Index(['Enginetype'], dtype='object')

In [28]:
# Numerical pipeline
num_pipeline = Pipeline([
    ('impute', SimpleImputer(strategy='mean')),
    ('winsorize', Winsorizer(capping_method='iqr', tail='both', fold=1.5)),
    ('scale', MinMaxScaler())
])

# Categorical pipeline
categ_pipeline = Pipeline([
    ('impute', SimpleImputer(strategy='most_frequent'),
    ('encode', OneHotEncoder( drop='first'))
])

In [29]:
# ColumnTransformer for all features
preprocess_pipeline = ColumnTransformer([
    ('numerical', num_pipeline, numeric_features),
    ('categorical', categ_pipeline, categorical_features)
])

In [30]:
# Fit the pipeline on X (features only)
clean = preprocess_pipeline.fit(X)

# Transform features and convert to DataFrame
Clean_data = pd.DataFrame(clean.transform(X))#, columns=clean.get_feature_names_out())


In [31]:
Clean_data.head()

,0,1,2,3,4,5,6,7
0,0.000000,0.37500,0.103108,0.347930,0.0,0.0,0.0,1.0
1,0.034286,0.40625,0.131581,0.403550,0.0,1.0,0.0,0.0
2,0.034286,0.40625,0.131581,0.394635,1.0,0.0,0.0,0.0
3,0.120000,0.40625,0.310106,0.408942,0.0,0.0,1.0,0.0
4,0.022857,0.40625,0.109265,0.384702,0.0,0.0,0.0,1.0


In [32]:
import os
os.getcwd()

'C:\\Users\\gauta\\OneDrive\\Desktop\\DataMites\\Study mate\\MLR'

In [41]:
# Save the preprocessing pipeline
joblib.dump(clean, 'preprocess_pipeline.pkl')


['akshya.pkl']

In [46]:

# Check the resulting DataFrame
print(Clean_data.head(10))


          0         1         2         3    4    5    6    7
0  0.000000  0.375000  0.103108  0.347930  0.0  0.0  0.0  1.0
1  0.034286  0.406250  0.131581  0.403550  0.0  1.0  0.0  0.0
2  0.034286  0.406250  0.131581  0.394635  1.0  0.0  0.0  0.0
3  0.120000  0.406250  0.310106  0.408942  0.0  0.0  1.0  0.0
4  0.022857  0.406250  0.109265  0.384702  0.0  0.0  0.0  1.0
5  0.120000  0.375000  0.303949  0.375000  0.0  0.0  0.0  1.0
6  0.034286  0.500000  0.131581  0.398383  0.0  0.0  0.0  0.0
7  0.074286  0.000000  0.067697  0.000000  0.0  0.0  0.0  0.0
8  0.074286  0.000000  0.067697  0.000000  0.0  0.0  1.0  0.0
9  0.177143  0.427083  0.358842  0.418340  0.0  0.0  0.0  1.0


In [35]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import train_test_split



In [36]:
# X_encoded = preprocessed features
# y = target variable

X_train, X_test, y_train, y_test = train_test_split(
    Clean_data, y, test_size=0.2, random_state=42
)


In [37]:
mlr = LinearRegression()
mlr.fit(X_train, y_train)


y_pred = mlr.predict(X_test)


In [42]:
X_test.shape

(17, 8)

In [43]:
y_test.head()

30    39.431235
0     53.700681
22    38.310606
31    39.431235
18    42.789909
Name: MPG, dtype: float64

In [45]:
y_pred

array([34.97066764, 43.8488357 , 36.61448337, 37.17209652, 40.38731707,
       40.13682801, 40.93334903, 29.0869202 , 42.71582082, 38.18139825,
       34.48734237, 33.51890745, 23.01628698, 38.55998251, 22.69604554,
       37.06820316, 22.69045627])

In [38]:
# R-squared
r2 = r2_score(y_test, y_pred)
print("R-squared:", r2)

# Mean Squared Error
mse = mean_squared_error(y_test, y_pred)
print("Mean Squared Error:", mse)

# Optional: Root Mean Squared Error
rmse = mse ** 0.5
print("RMSE:", rmse)


R-squared: 0.7862937502328178
Mean Squared Error: 21.01519695042328
RMSE: 4.584233518312879


In [39]:
import joblib

# Save preprocessing pipeline
joblib.dump(clean, 'preprocessing_pipeline.pkl')  # Saves in current working directory
preprocess_pipeline = joblib.load('preprocessing_pipeline.pkl')

In [40]:
# Assuming mlr is your trained LinearRegression model
joblib.dump(mlr, 'mlr_model.pkl')  # Saves in current working directory

mlr_model = joblib.load('mlr_model.pkl')



In [1]:
27.4**0.5

5.2345009313209605